# Swiggy MCP - an agent with a spending limit

Connecting Swiggy's official MCP server (`https://mcp.swiggy.com/food`) to a LangChain agent.

This is the first server in the course that is **remote, authenticated, and attached to a real account
that can spend real money**. That changes the job. The Reddit server read public listings; this one sees
my saved addresses, my order history, and my cart.

So the notebook is less about "how do I call a tool" and more about **what do I refuse to hand the model**.
Three module-2 ideas do the work:

| Problem | Module-2 tool |
|---|---|
| 4 of the 18 tools spend money | curate the tool list before `create_agent` |
| Every tool needs an `addressId` the model must not guess | runtime context (2.2) |
| Nothing stops a large cart | custom state + a `Command`-updating tool (2.2) |

In [1]:
import warnings

from dotenv import load_dotenv

load_dotenv()

# The checkpointer serialises our context dataclass on every step and pydantic grumbles
# about it. Harmless, and it drowns the output otherwise.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")

## No subprocess this time

The Reddit notebook needed the Windows Proactor event-loop fix because `transport: stdio` spawns a real
child process. This server is `streamable_http` - an HTTPS call to Swiggy's machines. No subprocess, no
event-loop policy, no `env` block. The awkward Windows preamble simply disappears.

What replaces it is authentication.

## The token

`swiggy_auth.py` (run once, separately) does OAuth 2.1 with PKCE against Swiggy: it registers a client,
opens a browser for phone + OTP, catches the redirect on `localhost:8765`, and exchanges the code for a
bearer token in `swiggy_token.json`.

That file is a **credential for a real Swiggy account** and is gitignored. It expires in 5 days
(`expires_in: 432000`); re-run `swiggy_auth.py` when calls start returning 401.

`MultiServerMCPClient` takes the token as an ordinary HTTP header. There is no Swiggy-specific LangChain
code anywhere in this notebook - it is just MCP over authenticated HTTP.

In [2]:
import json
from pathlib import Path

from langchain_mcp_adapters.client import MultiServerMCPClient

token = json.loads(Path("swiggy_token.json").read_text())

client = MultiServerMCPClient(
    {
        "swiggy": {
            "transport": "streamable_http",
            "url": "https://mcp.swiggy.com/food",
            "headers": {"Authorization": f"Bearer {token['access_token']}"},
        }
    }
)

## List the tools before writing a single prompt

Same rule as the Reddit notebook. Here it matters more: I need to know exactly which tools can move money
before deciding what the agent is allowed to hold.

In [3]:
mcp_tools = {tool.name: tool for tool in await client.get_tools()}

print(f"{len(mcp_tools)} tools")
for name in mcp_tools:
    print(" ", name)

18 tools
  get_addresses
  search_restaurants
  search_menu
  get_restaurant_menu
  get_food_cart
  update_food_cart
  flush_food_cart
  place_food_order
  fetch_food_coupons
  apply_food_coupon
  get_food_orders
  get_food_order_details
  track_food_order
  get_food_delivery_status
  report_error
  get_payment_options
  check_payment_status
  confirm_order


## Sort them by blast radius

This step has no equivalent in the Reddit notebook, and it is the whole point of this one.

`get_tools()` returns 18 tools. **Nothing obliges me to pass all 18 to `create_agent`.** The tool list is
an ordinary Python list, and I get to filter it.

- **READ** - searching, browsing, history, tracking. Cannot change anything.
- **CART** - mutating but reversible. `flush_food_cart` undoes all of it.
- **CHECKOUT** - places orders and moves money. **Never handed to the agent.**

Curation beats instruction. A system prompt saying "never place an order" is a request the model can be
argued out of. A tool that was never passed to `create_agent` is one the model cannot call, because it
does not know the tool exists and has no channel to invoke it. Its description never even enters the
prompt.

In [4]:
READ = [
    "search_restaurants",
    "search_menu",
    "get_restaurant_menu",
    "get_food_orders",
    "get_food_order_details",
    "track_food_order",
    "fetch_food_coupons",
]
CART = ["update_food_cart", "get_food_cart", "flush_food_cart", "apply_food_coupon"]
CHECKOUT = ["place_food_order", "get_payment_options", "check_payment_status", "confirm_order"]

print("withheld from the agent:")
for name in CHECKOUT:
    print(f"  {name}")

withheld from the agent:
  place_food_order
  get_payment_options
  check_payment_status
  confirm_order


## The addressId problem

Read the description Swiggy ships with `search_restaurants`:

> REQUIRED WORKFLOW: You MUST call get_addresses first to obtain a valid addressId... NEVER guess,
> invent, or use placeholder values like "default", "1", or "N/A".

Shouting in a docstring is a **prompt-level** guardrail, and it is the weakest kind there is. The model
is being trusted to choose correctly among nine saved addresses, and a wrong choice sends food to the
wrong city - this account has addresses in Delhi, Jaipur, Mumbai and Agra.

Module 2 has a better answer: resolve the address **in Python, once**, and inject it. The model never
receives the parameter, so it cannot fill it in wrongly.

## Whose instructions are these?

Before parsing anything, look at what Swiggy actually appends to a tool result:

```
TO PAY: Rs.143

Cart widget is displayed - do not repeat the cart items, prices, or totals in text.
A rich UI widget may be shown to the user with this data. Avoid restating everything
the widget already displays...
```

That is not data. Those are **instructions**, written for a different assistant - one running inside the
Swiggy app with a real UI widget to render. This notebook has no widget. But the text lands in a
ToolMessage all the same, and a ToolMessage carries the authority of the conversation.

An earlier run of this notebook obeyed them. The `OrderPlan` at the bottom came back with `lines=[]` and
`total_payable=0.0` from a cart that definitely had a dish in it - the model had been told not to repeat
cart items in text, so it did not. No error, no warning, just a confidently empty order.

This is the prompt-injection shape from module 1, arriving through a channel that feels like plumbing.
Nobody attacked anything here; Swiggy's copy is entirely well-intentioned. It still hijacked the output.

So `as_text` truncates at the first directive line. Wrapping an MCP tool is where you control its
description - and its **output** is worth the same scrutiny.

In [5]:
import re

# Swiggy appends advice for a *different* assistant to every tool result - one that renders
# rich widgets. Those lines are instructions, and our model reads them as such. See the
# "Whose instructions are these?" section below for what happens when they are left in.
DIRECTIVES = (
    "Cart widget is displayed",
    "NOTE: The cart widget",
    "A rich UI widget",
    "Some items have customisation",
    "Ask the user which",
    "To apply a coupon",
    "For items, image",
)


def as_text(result) -> str:
    """Join an MCP result's text blocks, dropping the server's trailing instructions to the model."""
    if isinstance(result, list):
        text = "\n".join(b.get("text", "") for b in result if isinstance(b, dict))
    else:
        text = str(result)

    lines = text.split("\n")
    for i, line in enumerate(lines):
        if any(d in line for d in DIRECTIVES):
            return "\n".join(lines[:i]).strip()
    return text.strip()


addresses = as_text(await mcp_tools["get_addresses"].ainvoke({}))

# Lines look like:  2. [Ghar] Name: street, city ... (ID: d803p26erajhob1ev1ng)
saved = re.findall(r"^\d+\.\s*\[([^\]]+)\].*?\(ID:\s*([A-Za-z0-9_-]+)\)", addresses, re.M)

for label, saved_id in saved:
    print(f"{label:>16}  {saved_id}")

           Other  213159933
            Ghar  d803p26erajhob1ev1ng
     mumbai home  215177010
            home  26406954
     rented flat  ctara26tpm3c02lap8o0
           Other  216693887
       New House  168102198
          Hostel  176136923
  mumbai address  260405738


In [6]:
ADDRESS_LABEL = "Ghar"

address_id = next(aid for label, aid in saved if label == ADDRESS_LABEL)
print(f"delivering to: {ADDRESS_LABEL} ({address_id})")

delivering to: Ghar (d803p26erajhob1ev1ng)


## Runtime context carries it

`context_schema` is the 2.2 pattern: a dataclass passed at invoke time, readable inside any tool through
`ToolRuntime`, and invisible to the model.

The budget lives here too. Context is for facts about *this run* that the model should obey but never
choose - which address, which spending cap.

In [7]:
from dataclasses import dataclass


@dataclass
class SwiggyContext:
    address_id: str
    address_label: str
    budget: float

## Wrapping the MCP tools

Each wrapper is a thin `@tool` that takes only the arguments the model should decide, pulls `address_id`
off `runtime.context`, and forwards to the real MCP tool.

Note what the docstrings are doing. Swiggy's own descriptions are written for a general assistant and are
full of instructions I do not want: `REQUIRED WORKFLOW` shouting, rich-widget hints, "ask the user which
address". Those arrive in my context verbatim. Wrapping replaces them with descriptions written for
*this* agent. Module 1's rule still holds: **the docstring is the prompt.**

In [8]:
from langchain.tools import tool, ToolRuntime


@tool
async def find_restaurants(query: str, runtime: ToolRuntime) -> str:
    """Search restaurants that deliver to the user. Returns names, ratings, ETA and restaurant IDs.

    Use for cuisine or restaurant-name queries, e.g. "biryani", "pizza", "KFC".
    """
    return as_text(
        await mcp_tools["search_restaurants"].ainvoke(
            {"addressId": runtime.context.address_id, "query": query}
        )
    )


@tool
async def find_dishes(query: str, runtime: ToolRuntime) -> str:
    """Search individual dishes across restaurants. Returns prices, menu item IDs and restaurant IDs.

    Use when the user names a dish rather than a place, e.g. "paneer butter masala".
    Both IDs in this result are required to add anything to the cart.
    """
    return as_text(
        await mcp_tools["search_menu"].ainvoke(
            {"addressId": runtime.context.address_id, "query": query}
        )
    )


@tool
async def past_orders(runtime: ToolRuntime) -> str:
    """List the user's recent Swiggy orders, newest first. Use to answer what they ordered before."""
    return as_text(
        await mcp_tools["get_food_orders"].ainvoke({"addressId": runtime.context.address_id})
    )


@tool
async def view_cart(runtime: ToolRuntime) -> str:
    """Show the current cart contents and the real total payable."""
    return as_text(
        await mcp_tools["get_food_cart"].ainvoke({"addressId": runtime.context.address_id})
    )


@tool
async def empty_cart() -> str:
    """Remove everything from the cart."""
    return as_text(await mcp_tools["flush_food_cart"].ainvoke({}))

## Proof: the parameter is gone

`args` is the schema the model actually sees. `addressId` is not in it - so "never guess the addressId"
stops being a hope and becomes a fact about the interface.

In [9]:
for t in (find_restaurants, find_dishes, past_orders):
    print(f"{t.name:20} {list(t.args)}")

find_restaurants     ['query']
find_dishes          ['query']
past_orders          []


## The budget guard, in state

`update_food_cart` is the one tool here with a side effect worth guarding. The guard has two halves, and
the second exists because of something a probe revealed.

Search listed a dish at **Rs.130**. The cart charged **Rs.119** for it, then added Rs.23.53 tax, for a real
payable of **Rs.143**. The number the model reads out of a search result is an estimate, and taxes and
fees are invisible until the cart exists.

So:

1. **Pre-check on the estimate** - refuse before calling Swiggy if the model's own arithmetic already
   breaks the cap. Nothing mutates, so there is nothing to undo.
2. **Re-read the server's total afterwards** - parse the real `TO PAY` and write *that* into state.

The model proposes; the server is authoritative. `Command` writes the true figure into `cart_total`, so
the next call guards against reality rather than against the model's guess.

In [10]:
from langchain.agents import AgentState


class SwiggyState(AgentState):
    cart_total: float
    restaurant: str

In [11]:
from langchain.messages import ToolMessage
from langgraph.types import Command


@tool
async def add_to_cart(
    restaurant_id: str,
    restaurant_name: str,
    menu_item_id: str,
    quantity: int,
    price_each: float,
    runtime: ToolRuntime,
) -> Command | str:
    """Add a dish to the cart. Requires restaurant_id and menu_item_id from find_dishes.

    price_each is the listed rupee price of one unit, used for a budget pre-check.
    """
    budget = runtime.context.budget
    estimate = runtime.state.get("cart_total", 0.0) + price_each * quantity

    if estimate > budget:
        return (
            f"REFUSED: that would reach about INR {estimate:.0f}, over the INR {budget:.0f} "
            f"budget for this cart. Nothing was added. Suggest something cheaper."
        )

    await mcp_tools["update_food_cart"].ainvoke(
        {
            "addressId": runtime.context.address_id,
            "restaurantId": restaurant_id,
            "cartItems": [{"menu_item_id": menu_item_id, "quantity": quantity}],
        }
    )

    # restaurantName is echoed straight back - the cart API does not always report it itself.
    cart = as_text(
        await mcp_tools["get_food_cart"].ainvoke(
            {"addressId": runtime.context.address_id, "restaurantName": restaurant_name}
        )
    )
    match = re.search(r"TO PAY:\s*.?([\d,.]+)", cart)
    actual = float(match.group(1).replace(",", "")) if match else estimate

    note = "" if actual <= budget else f" WARNING: this is over the INR {budget:.0f} budget."
    return Command(
        update={
            "cart_total": actual,
            "restaurant": restaurant_name,
            "messages": [
                ToolMessage(
                    f"Added. Real total payable is INR {actual:.0f} including tax.{note}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

## Build the agent

Past this line there is nothing MCP-specific and nothing Swiggy-specific. Same `create_agent` as module 1,
plus the two schemas.

Note that the system prompt says nothing about not placing orders. It does not need to - there is no
checkout tool in the list. Guardrails that live in code do not have to be repeated in prose.

In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

MODEL = "openrouter:gemini-3.5-flash-lite"

agent = create_agent(
    model=MODEL,
    tools=[find_restaurants, find_dishes, past_orders, view_cart, empty_cart, add_to_cart],
    state_schema=SwiggyState,
    context_schema=SwiggyContext,
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You order food on Swiggy for the user. Always search before recommending - never invent "
        "restaurants, dishes or prices. When adding to the cart you must use the exact restaurant "
        "and menu item IDs returned by the search tools. Be concise."
    ),
)

context = SwiggyContext(address_id=address_id, address_label=ADDRESS_LABEL, budget=300.0)
config = {"configurable": {"thread_id": "dinner"}}

## Read-only first

In [13]:
response = await agent.ainvoke(
    {"messages": [("user", "What have I been ordering lately?")]},
    config,
    context=context,
)

print(response["messages"][-1].content)

Here are your recent orders, starting with the most recent:

1. **The Belgian Waffle Co.** (June 15 & May 26) – Mini Waffle box of 4 - Premium Assorted
2. **Shawarmajaan** (March 20) – Chicken Kuboos Shawarma - Classic & Falafel Shots
3. **Pizza Hut** (January 25) – Veggie Feast pizza
4. **Chhabras Pure Veg Restaurant** (September 18) – Malai Kofta & Mushroom Masala

Would you like


## Now let it spend, within the cap

Rs.300 budget. A cheap dish should go through, and state should come back holding the server's real total
rather than the model's estimate.

In [14]:
response = await agent.ainvoke(
    {"messages": [("user", "Find me a cheap paneer dish under Rs.150 and add one to my cart.")]},
    config,
    context=context,
)

print(response["messages"][-1].content)
print("\ncart_total in state:", response.get("cart_total"))

Added **

cart_total in state: 132.0


## Now let it try to overspend

Same thread, so the cart total already in state carries forward. Asking for something expensive should be
refused by the pre-check - and the refusal happens in Python, before any request reaches Swiggy.

In [15]:
response = await agent.ainvoke(
    {"messages": [("user", "Actually add a large biryani too, around Rs.400.")]},
    config,
    context=context,
)

print(response["messages"][-1].content)
print("\ncart_total in state:", response.get("cart_total"))

I'

cart_total in state: 183.0


## The trace

Same shape as every agent so far: Human, AI with `tool_calls`, ToolMessage, AI. The interesting part is
that the ToolMessage for a refusal looks identical to one for a success - the guard is not an exception,
it is just a string the model has to deal with.

In [16]:
for message in response["messages"][-6:]:
    message.pretty_print()

================================= Tool Message =================================
Name: add_to_cart

REFUSED: that would reach about INR 301, over the INR 300 budget for this cart. Nothing was added. Suggest something cheaper.
================================== Ai Message ==================================
Tool Calls:
  find_dishes (call_2902092)
 Call ID: call_2902092
  Args:
    query: paneer curry
================================= Tool Message =================================
Name: find_dishes

Found 10 menu items for "paneer curry":
1. Palak Paneer - RICE Meal — ₹338 | Veg | 5.0★ | GharSe - Homestyle & Healthy Tiffins (restaurantId: 1155522) (ID: 184621338) [has addons]
2. Paneer Butter Masala — ₹259 | Veg | 3.7★ | Apni Rasoi (restaurantId: 966660) (ID: 152725450) [has addons]
3. Paneer Lababdar — ₹229 | Veg | 5.0★ | BOX8 - Desi Meals (restaurantId: 449572) (ID: 83646767)
4. Matar Paneer — ₹259 | Veg | 4.0★ | Apni Rasoi (restaurantId: 966660) (ID: 152725460) [has addons]
5. Paneer 

In [17]:
# What did the tool results cost us in context?
from langchain.messages import AIMessage

tool_calls = sum(len(m.tool_calls) for m in response["messages"] if isinstance(m, AIMessage))
tool_chars = sum(len(str(m.content)) for m in response["messages"] if isinstance(m, ToolMessage))

print(f"messages:   {len(response['messages'])}")
print(f"tool calls: {tool_calls}")
print(f"tool result characters in context: {tool_chars:,}")

messages:   34
tool calls: 14
tool result characters in context: 7,502


## A checkable handoff

The agent stops at a plan. Rather than trusting prose, ask for module 1's structured output so the final
object is typed and inspectable - and so a human can approve a real object before anything is paid for.

Watch where the restaurant name comes from. The cart API does not reliably return it, and the first run of
this notebook produced `restaurant='Swiggy'` - the model filling a gap with something plausible and wrong.
The name is not in the cart, so it has to come from **state**, where `add_to_cart` recorded it at the
moment it knew. A typed field does not stop a model inventing a value; it only makes the invention easy
to spot. Feeding the known fact in is what fixes it.

In [18]:
from pydantic import BaseModel, Field


class CartLine(BaseModel):
    dish: str
    restaurant: str
    quantity: int
    price_each: float


class OrderPlan(BaseModel):
    """A cart ready for a human to approve."""

    restaurant: str = Field(description="Restaurant the order is from")
    lines: list[CartLine]
    total_payable: float = Field(description="Real total including tax, taken from the cart")
    within_budget: bool


planner = create_agent(
    model=MODEL,
    tools=[view_cart],
    context_schema=SwiggyContext,
    response_format=OrderPlan,
    system_prompt=(
        "Summarise the user's current Swiggy cart exactly as the cart tool reports it. "
        "Never invent a restaurant, dish or price."
    ),
)

known_restaurant = response["restaurant"]  # recorded in state by add_to_cart
budget = context.budget

plan = (
    await planner.ainvoke(
        {
            "messages": [
                (
                    "user",
                    f"Summarise my cart. It is from {known_restaurant}, "
                    f"and the budget is INR {budget:.0f}.",
                )
            ]
        },
        context=context,
    )
)["structured_response"]

plan

OrderPlan(restaurant='Biryani Blues', lines=[CartLine(dish='Veg Dum Biryani Bowl', restaurant='Biryani Blues', quantity=1, price_each=129.0)], total_payable=183.0, within_budget=True)

## Clean up

The cart is real. Leaving it full means the Swiggy app shows a pending cart on the phone.

In [19]:
print(as_text(await mcp_tools["flush_food_cart"].ainvoke({})))

Flushed Food cart successfully


## What to remember

- **`streamable_http` needs no subprocess.** No Proactor fix, no `env` block - the awkward Windows
  preamble from the stdio notebook is simply absent. Authentication replaces it, as an ordinary HTTP
  header.
- **`get_tools()` returns a list, and lists can be filtered.** Four of Swiggy's eighteen tools move money.
  They were never passed to `create_agent`, so their descriptions never entered the prompt and the model
  has no channel to call them. That is stronger than any instruction, because there is nothing to argue
  the model out of.
- **A server shouting "NEVER guess this parameter" is a design smell.** It asks the prompt to do a job the
  interface should do. Resolving `addressId` in Python and injecting it through `context_schema` removes
  the parameter from the model's schema entirely - verified with `tool.args`.
- **Wrapping an MCP tool rewrites its prompt.** Third-party descriptions arrive verbatim in your context,
  complete with instructions aimed at somebody else's assistant. A thin `@tool` wrapper is where you take
  that back.
- **Tool *output* is a prompt too, and this one bit.** Swiggy appends "do not repeat the cart items,
  prices, or totals in text" to every cart result - sound advice for an assistant with a UI widget, and
  poison for one without. The agent obeyed and produced an empty `OrderPlan` from a full cart, with no
  error. Well-intentioned copy from a first-party server did that; imagine a server that meant harm.
  Sanitise what comes back, not just what goes out.
- **The model's numbers are estimates; the server's are facts.** Listed Rs.130, cart line Rs.119, real
  payable Rs.143 after tax. Guard on the estimate to avoid the mutation, then write the server's figure
  into state.
- **Guard before the side effect, not after.** The over-budget refusal returns before `update_food_cart`
  is ever called, so there is nothing to roll back.
- **A typed field does not prevent invention, it only exposes it.** `OrderPlan.restaurant` came back as
  `'Swiggy'` on the first run, because the cart API had not reported a name and the model filled the hole
  with something plausible. Pydantic validated it happily - it was a perfectly good string. The fix was to
  carry the real name in state from the moment `add_to_cart` knew it. Structured output constrains shape,
  never truth.
- **Swiggy's tool output is pre-summarised text, not raw JSON.** Roughly 1.5k characters per call against
  the Reddit server's full listings. A well-built MCP server trims for you - but confirm it by measuring
  rather than assuming.
- **This token is a real credential.** Gitignored, 5-day expiry, and it can read every address and order
  on the account. Treat a bearer token in a notebook with the same care as a password.